### Preprocessing
1) Loads both validation and test datasets
2) Checks the entity distribution 
3) Inserts emails into the datasets
4) Saves the new modified datasets in the "datasets" folder 

In [35]:
import re
import json
import random
from copy import deepcopy
from pathlib import Path


DOMAINS = ['gmail.com', 'yahoo.com', 'outlook.com', 'live.com', 'hotmail.com']

SEPARATORS = ['-', '_', '.', '']


def load_data():
    with open('../datasets/data.json', 'r', encoding='utf-8') as file:
        train_data = json.load(file)

    print(f"Data loaded")
    return train_data

train_data = load_data()

def analyze_distribution(data):
    """ Checks the tag distributions in the dataset """
    counts = {}
    for entry in data:
        for tag in entry['ner_tags']:
            if tag not in counts:
                counts[tag] = 0
            counts[tag] += 1
    print()
    print("\033[1mTag Distribution\033[0m")
    total = sum(counts.values())

    for tag, count in counts.items():
        print(f"{tag} : {count} ({count / total * 100 :.2f}%)")

    if 'B-EMAIL' not in counts:
        print("No emails in dataset, imputation required.")


def extract_names(data):
    """ Extracts names from the given dataset and returns two lists, first_names and last_names """

    first_names = set()
    last_names = set()
    ignore = {'is', 'poor', 'a', 'the', 'that'}

    for entry in data:
        tags = entry['ner_tags']
        tokens = entry['tokens']

        current_name = []
        for token, tag in zip(tokens, tags):
            token = token.lower()
            if tag == 'B-PER':
                current_name = [token]

            elif tag == 'I-PER':
                current_name.append(token)

            else:
                if current_name:                    # Condition to jump into if name has ended
                    if current_name[0] not in ignore:
                        first_names.add(current_name[0])
                        for name in current_name[1:]:
                            if name not in ignore:
                                last_names.add(name)
                current_name = []

        if current_name:                       # To resolve edge case where sequence ends with name 
            if current_name[0] not in ignore:
                first_names.add(current_name[0])
                for name in current_name[1:]:
                    if name not in ignore:
                        last_names.add(name)

    return list(first_names), list(last_names)                      

first_names, last_names = extract_names(train_data)

def clean_name(name):
    return re.sub(r'[^a-z]', '', name.lower())

def generate_email(first, last):
    """ Generates a realistic synthetic email address """
    first = clean_name(first)
    last = clean_name(last)
    if not first:
        first = random.choice(first_names)
    if not last:
        last = random.choice(last_names)
    sep   = random.choice(SEPARATORS)
    domain = random.choice(DOMAINS)
    randnum = random.randint(1, 999)
    pattern = random.choice([
        f"{first}{sep}{last}",
        f"{first[0]}{sep}{last}",
        f"{first}{sep}{last[0]}",
        f"{first}",
        f"{first}{randnum}",
        f"{last}{randnum}",
        f"{first}{sep}{last}{randnum}",
        f"{first}{randnum}{last}",
        f"{last}{first}", 
        f"{last}{sep}{first}",
        f"{first[0]}{last}{randnum}",       
        f"{first}{sep}{last[0]}{randnum}"
    ])
 
    return f"{pattern}@{domain}"





def insert_email_in_entry(entry, insertion_prob=0.4):
    lang = entry['lang']
    tags = entry["ner_tags"]
    tokens = entry["tokens"]

    new_tokens = []
    new_tags = []

    current_name = []
    for token, tag in zip(tokens, tags):
        if tag == 'B-PER':
            current_name = [token]
        
        elif tag == 'I-PER':
            current_name.append(token)
        
        else:
            if current_name:
                new_tokens.extend(current_name)
                new_tags.extend(['B-PER'] + ['I-PER'] * (len(current_name) - 1))

                if random.random() < insertion_prob:   # Inserting in some entries 

                    if random.random() < 0.5:          # 50% chance of generating email with actual first and last name of person 
                        email = generate_email(current_name[0].lower(), current_name[-1].lower())

                    else:                              # Else pick randomly from among the first and last names lists
                        email = generate_email(random.choice(first_names), random.choice(last_names))
                    new_tags.append('B-EMAIL')
                    new_tokens.append(email)

            current_name = []

            new_tags.append(tag)     # Append remaining tokens as is
            new_tokens.append(token)

    if current_name:
        new_tokens.extend(current_name)
        new_tags.extend(['B-PER'] + ['I-PER'] * (len(current_name) - 1))

        if random.random() < insertion_prob:   # Inserting in some entries 

            if random.random() < 0.5:          # 50% chance of generating email with actual first and last name of person 
                email = generate_email(current_name[0].lower(), current_name[-1].lower())

            else:                               # Else pick randomly from among the first and last names lists
                email = generate_email(random.choice(first_names), random.choice(last_names))

            new_tags.append('B-EMAIL')
            new_tokens.append(email)

    new_sequence = ' '.join(new_tokens)  # Creating new sequence

    return {'lang': lang, 'ner_tags': new_tags, 'sequence': new_sequence, 'tokens': new_tokens}


def insert_emails(data):
    modified_data = []
    for entry in data:
        modified_data.append(insert_email_in_entry(entry))
    return modified_data

def create_agumented_dataset():
    new_train_data = insert_emails(train_data)
    with open('../datasets/train_data_modified.json', 'w', encoding='utf-8') as file:
        json.dump(new_train_data, file, indent=4)
    return new_train_data


def main(): 
    analyze_distribution(train_data)
    new_train_data = create_agumented_dataset()
    analyze_distribution(new_train_data)
    print("\n\033[1mAugmented Dataset Analysis:\033[0m")
    for i in range(10):
        emails = []
        tokens = new_train_data[i]['tokens']
        tags = new_train_data[i]['ner_tags']
        for j, tag in enumerate(tags):
            if tag == 'B-EMAIL':
                emails.append(tokens[j])
        print(f"\nEmail(s) added: {emails}")
        print(f"New Sequence: {new_train_data[i]['sequence']}")

main()
        





Data loaded

Tag Distribution
O : 639055 (90.16%)
B-PER : 40264 (5.68%)
I-PER : 29466 (4.16%)
No emails in dataset, imputation required.

Tag Distribution
O : 639055 (88.18%)
B-PER : 40253 (5.55%)
I-PER : 29457 (4.06%)
B-EMAIL : 15975 (2.20%)

Augmented Dataset Analysis:

Email(s) added: ['joe824montana@gmail.com', 'marlowe921@outlook.com']
New Sequence: Since then , only Terry Bradshaw in 147 games , Joe Montana joe824montana@gmail.com in 139 games , and Tom Brady marlowe921@outlook.com in 131 games have reached 100 wins more quickly .

Email(s) added: []
New Sequence: He was portrayed by Anthony Perkins in the 1960 version of " Psycho " directed by Alfred Hitchcock and the " Psycho " franchise .

Email(s) added: ['lorrie780@hotmail.com']
New Sequence: The egg eventually hatches , revealing a baby Sharptooth lorrie780@hotmail.com .

Email(s) added: []
New Sequence: In the video Kelis is walking down a street in a large space suit-style coat , then singing to the stars .

Email(s) adde